In [ ]:
# idea:
# RoBERTa shared encoder
# hatespeech distribution head     3 classes
# insult distribution head         5 classes
# dehumanize distribution head     5 classes
# violence distribution head       5 classes
# genocide distribution head       5 classes

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader

from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, AutoConfig, DataCollatorWithPadding
from tqdm.auto import tqdm

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 12.1


device(type='cuda')

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
ann = pd.read_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv")

print("Loaded:", ann.shape)
print("Unique comments:", ann["comment_id"].nunique())
print(ann["split"].value_counts())

Loaded: (49433, 53)
Unique comments: 19761
split
train         34413
test           7713
validation     7307
Name: count, dtype: int64


In [5]:
task_specs = {
    "hatespeech": [0, 1, 2],
    "insult": [0, 1, 2, 3, 4],
    "dehumanize": [0, 1, 2, 3, 4],
    "violence": [0, 1, 2, 3, 4],
    "genocide": [0, 1, 2, 3, 4],
}

task_names = list(task_specs.keys())
task_names

['hatespeech', 'insult', 'dehumanize', 'violence', 'genocide']

In [6]:
def make_distribution(group, label_col, label_values):
    counts = group[label_col].value_counts(normalize=True)

    return pd.Series({
        f"{label_col}_{label}_prob": counts.get(label, 0.0)
        for label in label_values
    })

In [7]:
metadata = (
    ann.groupby("comment_id")
    .agg(
        text_clean=("text_clean", "first"),
        split=("split", "first"),
        target_type=("target_type", "first"),
        is_women_targeted=("is_women_targeted", "max"),
        is_immigrant_targeted=("is_immigrant_targeted", "max"),
        annotation_count=("annotator_id", "count"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
)

comment_df = metadata.copy()

for task, label_values in task_specs.items():
    dist = (
        ann.groupby("comment_id")
        .apply(lambda g, task=task, label_values=label_values: make_distribution(g, task, label_values))
        .reset_index()
    )

    comment_df = comment_df.merge(dist, on="comment_id", how="left")
    print(task, dist.shape)

print("Comment-level shape:", comment_df.shape)
comment_df.head()

hatespeech (19761, 4)
insult (19761, 6)
dehumanize (19761, 6)
violence (19761, 6)
genocide (19761, 6)
Comment-level shape: (19761, 31)


,comment_id,text_clean,split,target_type,is_women_targeted,is_immigrant_targeted,annotation_count,unique_annotators,hatespeech_0_prob,hatespeech_1_prob,...,violence_0_prob,violence_1_prob,violence_2_prob,violence_3_prob,violence_4_prob,genocide_0_prob,genocide_1_prob,genocide_2_prob,genocide_3_prob,genocide_4_prob
0,6,First off you look cool as fuck! Anyway if we ...,train,women_only,1,0,2,2,1.000000,0.0,...,0.0,0.5,0.0,0.000000,0.500000,0.0,1.0,0.000000,0.0,0.000000
1,7,\*points to posters asking for palestinian rig...,test,immigrant_only,0,1,2,2,0.500000,0.0,...,0.5,0.0,0.0,0.000000,0.500000,0.5,0.0,0.000000,0.0,0.500000
2,10,"They'll come back in your plan, also. Plus we ...",train,immigrant_only,0,1,3,3,0.333333,0.0,...,0.0,0.0,0.0,0.333333,0.666667,0.0,0.0,0.333333,0.0,0.666667
3,11,"eat my fuck, bitch",validation,women_only,1,0,2,2,0.000000,0.5,...,0.5,0.5,0.0,0.000000,0.000000,1.0,0.0,0.000000,0.0,0.000000
4,12,She ugly anyways,train,women_only,1,0,2,2,0.500000,0.0,...,0.5,0.5,0.0,0.000000,0.000000,1.0,0.0,0.000000,0.0,0.000000


In [8]:
for task, label_values in task_specs.items():
    cols = [f"{task}_{label}_prob" for label in label_values]
    sums = comment_df[cols].sum(axis=1)

    print(task)
    print("min:", sums.min())
    print("max:", sums.max())
    print("mean:", sums.mean())
    print()

hatespeech
min: 1.0
max: 1.0
mean: 1.0

insult
min: 0.9999999999999999
max: 1.0
mean: 1.0

dehumanize
min: 0.9999999999999999
max: 1.0
mean: 1.0

violence
min: 0.9999999999999999
max: 1.0000000000000002
mean: 1.0

genocide
min: 0.9999999999999998
max: 1.0
mean: 1.0



In [9]:
train_df = comment_df[comment_df["split"] == "train"].copy()
val_df = comment_df[comment_df["split"] == "validation"].copy()
test_df = comment_df[comment_df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (13832, 31)
Validation: (2964, 31)
Test: (2965, 31)


In [10]:
label_cols = []

for task, label_values in task_specs.items():
    label_cols.extend([f"{task}_{label}_prob" for label in label_values])

hf_cols = ["text_clean"] + label_cols

train_dataset = Dataset.from_pandas(train_df[hf_cols].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[hf_cols].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[hf_cols].reset_index(drop=True))

In [11]:
model_name = "roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [12]:
def tokenize(batch):
    return tokenizer(
        batch["text_clean"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/13832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

In [13]:
def add_task_labels(batch):
    n = len(batch["text_clean"])

    for task, label_values in task_specs.items():
        batch[f"{task}_labels"] = [
            [
                float(batch[f"{task}_{label}_prob"][i])
                for label in label_values
            ]
            for i in range(n)
        ]

    return batch

train_dataset = train_dataset.map(add_task_labels, batched=True)
val_dataset = val_dataset.map(add_task_labels, batched=True)
test_dataset = test_dataset.map(add_task_labels, batched=True)

Map:   0%|          | 0/13832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

In [14]:
columns_to_keep = ["input_ids", "attention_mask"] + [
    f"{task}_labels" for task in task_names
]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [15]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

In [16]:
class RobertaGlobalMultidimensionalSoftLabelModel(nn.Module):
    def __init__(self, model_name, task_specs, dropout_prob=0.2):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)

        self.task_heads = nn.ModuleDict({
            task: nn.Linear(hidden_size, len(label_values))
            for task, label_values in task_specs.items()
        })

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)

        logits = {
            task: head(pooled)
            for task, head in self.task_heads.items()
        }

        return logits

In [17]:
model = RobertaGlobalMultidimensionalSoftLabelModel(
    model_name=model_name,
    task_specs=task_specs,
    dropout_prob=0.2
)

model.to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaGlobalMultidimensionalSoftLabelModel(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True

In [18]:
def compute_batch_loss(model, batch):
    labels = {}

    for task in task_names:
        labels[task] = batch.pop(f"{task}_labels").float().to(device)

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    task_losses = {}

    for task in task_names:
        logits = outputs[task]
        task_labels = labels[task]

        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

        loss = torch.nn.functional.kl_div(
            log_probs,
            task_labels,
            reduction="batchmean"
        )

        task_losses[task] = loss

    total_loss = torch.stack(list(task_losses.values())).mean()

    return total_loss, task_losses

In [19]:
def distribution_metrics(probs, labels):
    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    hard_accuracy = (
        probs.argmax(axis=1) == labels.argmax(axis=1)
    ).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    if np.std(pred_entropy) == 0 or np.std(gold_entropy) == 0:
        entropy_corr = np.nan
    else:
        entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    return {
        "kl_divergence": float(kl),
        "js_divergence": float(js),
        "hard_accuracy": float(hard_accuracy),
        "entropy_correlation": float(entropy_corr)
    }

In [20]:
def evaluate_model(model, loader):
    model.eval()

    all_probs = {
        task: []
        for task in task_names
    }

    all_labels = {
        task: []
        for task in task_names
    }

    total_losses = []
    task_loss_store = {
        task: []
        for task in task_names
    }

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            labels = {}

            for task in task_names:
                labels[task] = batch.pop(f"{task}_labels").float()

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            losses = []

            for task in task_names:
                logits = outputs[task]
                task_labels = labels[task]

                log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

                loss = torch.nn.functional.kl_div(
                    log_probs,
                    task_labels,
                    reduction="batchmean"
                )

                losses.append(loss)
                task_loss_store[task].append(loss.item())

                probs = torch.nn.functional.softmax(logits, dim=-1)

                all_probs[task].append(probs.cpu().numpy())
                all_labels[task].append(task_labels.cpu().numpy())

            total_loss = torch.stack(losses).mean()
            total_losses.append(total_loss.item())

    rows = []

    for task in task_names:
        probs = np.vstack(all_probs[task])
        labels = np.vstack(all_labels[task])

        metrics = distribution_metrics(probs, labels)

        row = {
            "task": task,
            "eval_loss": float(np.mean(task_loss_store[task]))
        }

        row.update(metrics)
        rows.append(row)

    results_df = pd.DataFrame(rows)

    macro_row = {
        "task": "MACRO_AVERAGE",
        "eval_loss": float(np.mean(total_losses)),
        "kl_divergence": results_df["kl_divergence"].mean(),
        "js_divergence": results_df["js_divergence"].mean(),
        "hard_accuracy": results_df["hard_accuracy"].mean(),
        "entropy_correlation": results_df["entropy_correlation"].mean(),
    }

    results_df = pd.concat(
        [results_df, pd.DataFrame([macro_row])],
        ignore_index=True
    )

    return results_df

In [21]:
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 6
PATIENCE = 1

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [22]:
output_dir = "../models/roberta_global_multidimensional_soft_labels_1"
best_model_dir = os.path.join(output_dir, "best_model")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(best_model_dir, exist_ok=True)

history_rows = []

best_val_js = float("inf")
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()

    train_losses = []
    train_task_losses = {
        task: []
        for task in task_names
    }

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")

    for batch in progress:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        loss, task_losses = compute_batch_loss(model, batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        train_losses.append(loss.item())

        for task in task_names:
            train_task_losses[task].append(task_losses[task].item())

        progress.set_postfix({
            "train_loss": np.mean(train_losses)
        })

    train_loss = float(np.mean(train_losses))

    val_results = evaluate_model(model, val_loader)
    val_macro = val_results[val_results["task"] == "MACRO_AVERAGE"].iloc[0]

    val_macro_js = float(val_macro["js_divergence"])
    val_macro_kl = float(val_macro["kl_divergence"])
    val_macro_acc = float(val_macro["hard_accuracy"])
    val_macro_entropy = float(val_macro["entropy_correlation"])

    print("\n" + "=" * 80)
    print(f"Epoch {epoch}")
    print(f"Train loss: {train_loss:.6f}")
    print("Validation metrics:")
    print(val_results.to_string(index=False))
    print("=" * 80 + "\n")

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_macro_js": val_macro_js,
        "val_macro_kl": val_macro_kl,
        "val_macro_accuracy": val_macro_acc,
        "val_macro_entropy_correlation": val_macro_entropy
    }

    for task in task_names:
        row[f"train_{task}_loss"] = float(np.mean(train_task_losses[task]))

    history_rows.append(row)

    improved = val_macro_js < best_val_js

    if improved:
        best_val_js = val_macro_js
        epochs_without_improvement = 0

        torch.save(model.state_dict(), os.path.join(best_model_dir, "pytorch_model.bin"))
        tokenizer.save_pretrained(best_model_dir)

        print(f"New best model saved. Best validation macro JS: {best_val_js:.6f}")

    else:
        epochs_without_improvement += 1
        print(f"No improvement. Patience counter: {epochs_without_improvement}/{PATIENCE}")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    torch.cuda.empty_cache()

Epoch 1/6:   0%|          | 0/1729 [00:00<?, ?it/s]


Epoch 1
Train loss: 0.870045
Validation metrics:
         task  eval_loss  kl_divergence  js_divergence  hard_accuracy  entropy_correlation
   hatespeech   0.520835       0.518972       0.137037       0.756073             0.275741
       insult   0.989563       0.989335       0.264065       0.425439             0.097769
   dehumanize   1.092837       1.093592       0.297409       0.368084             0.125665
     violence   0.847399       0.844658       0.222515       0.625506             0.200772
     genocide   0.609263       0.608278       0.154675       0.797571             0.189987
MACRO_AVERAGE   0.811980       0.810967       0.215140       0.594534             0.177987

New best model saved. Best validation macro JS: 0.215140


Epoch 2/6:   0%|          | 0/1729 [00:00<?, ?it/s]


Epoch 2
Train loss: 0.797595
Validation metrics:
         task  eval_loss  kl_divergence  js_divergence  hard_accuracy  entropy_correlation
   hatespeech   0.511483       0.509715       0.135566       0.754049             0.272153
       insult   0.991272       0.990905       0.256047       0.439609             0.076240
   dehumanize   1.096545       1.097463       0.290815       0.368421             0.119233
     violence   0.839942       0.837525       0.223374       0.604926             0.208883
     genocide   0.607283       0.606461       0.155364       0.799258             0.190188
MACRO_AVERAGE   0.809305       0.808414       0.212233       0.593252             0.173339

New best model saved. Best validation macro JS: 0.212233


Epoch 3/6:   0%|          | 0/1729 [00:00<?, ?it/s]


Epoch 3
Train loss: 0.755484
Validation metrics:
         task  eval_loss  kl_divergence  js_divergence  hard_accuracy  entropy_correlation
   hatespeech   0.534381       0.531763       0.130153       0.767544             0.279912
       insult   1.004520       1.004507       0.257239       0.464912             0.091681
   dehumanize   1.112000       1.112669       0.290269       0.396424             0.126028
     violence   0.867248       0.863035       0.214973       0.618084             0.201300
     genocide   0.635479       0.633787       0.147229       0.798246             0.175095
MACRO_AVERAGE   0.830726       0.829152       0.207973       0.609042             0.174803

New best model saved. Best validation macro JS: 0.207973


Epoch 4/6:   0%|          | 0/1729 [00:00<?, ?it/s]


Epoch 4
Train loss: 0.715306
Validation metrics:
         task  eval_loss  kl_divergence  js_divergence  hard_accuracy  entropy_correlation
   hatespeech   0.523346       0.520525       0.132227       0.762146             0.255780
       insult   0.997646       0.997342       0.254170       0.462213             0.077021
   dehumanize   1.111669       1.112008       0.293403       0.352227             0.125569
     violence   0.863110       0.858242       0.213442       0.625169             0.196373
     genocide   0.623745       0.622295       0.151669       0.790823             0.184934
MACRO_AVERAGE   0.823903       0.822082       0.208982       0.598516             0.167935

No improvement. Patience counter: 1/1
Early stopping triggered.


In [23]:
best_model_path = os.path.join(best_model_dir, "pytorch_model.bin")

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)

print("Loaded best model from:", best_model_path)
print("Best validation macro JS:", best_val_js)

/tmp/ipykernel_11711/2985465439.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=device))


Loaded best model from: ../models/roberta_global_multidimensional_soft_labels_1/best_model/pytorch_model.bin
Best validation macro JS: 0.20797250270843506


In [24]:
val_results_df = evaluate_model(model, val_loader)
test_results_df = evaluate_model(model, test_loader)

print("Final validation metrics:")
display(val_results_df)

print("Final test metrics:")
display(test_results_df)

Final validation metrics:


,task,eval_loss,kl_divergence,js_divergence,hard_accuracy,entropy_correlation
0,hatespeech,0.534381,0.531763,0.130153,0.767544,0.279912
1,insult,1.004520,1.004507,0.257239,0.464912,0.091681
2,dehumanize,1.112000,1.112669,0.290269,0.396424,0.126028
3,violence,0.867248,0.863035,0.214973,0.618084,0.201300
4,genocide,0.635479,0.633787,0.147229,0.798246,0.175095
5,MACRO_AVERAGE,0.830726,0.829152,0.207973,0.609042,0.174803


Final test metrics:


,task,eval_loss,kl_divergence,js_divergence,hard_accuracy,entropy_correlation
0,hatespeech,0.570062,0.568457,0.137230,0.754806,0.252359
1,insult,0.963113,0.963836,0.248901,0.476560,0.093876
2,dehumanize,1.130585,1.129273,0.292893,0.395278,0.138657
3,violence,0.883457,0.885017,0.219791,0.607420,0.174365
4,genocide,0.632549,0.634341,0.144865,0.806071,0.176002
5,MACRO_AVERAGE,0.835953,0.836185,0.208736,0.608027,0.167052


In [25]:
final_model_dir = os.path.join(output_dir, "final_model")

os.makedirs(final_model_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(final_model_dir, "pytorch_model.bin"))
tokenizer.save_pretrained(final_model_dir)

print("Saved final model to:", final_model_dir)

Saved final model to: ../models/roberta_global_multidimensional_soft_labels_1/final_model


In [27]:
os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_global_multidimensional_soft_labels_1_results.txt"
history_path = "../results/tables/roberta_global_multidimensional_soft_labels_1_training_history.csv"
val_metrics_path = "../results/tables/roberta_global_multidimensional_soft_labels_1_validation_metrics.csv"
test_metrics_path = "../results/tables/roberta_global_multidimensional_soft_labels_1_test_metrics.csv"

history_df = pd.DataFrame(history_rows)

history_df.to_csv(history_path, index=False)
val_results_df.to_csv(val_metrics_path, index=False)
test_results_df.to_csv(test_metrics_path, index=False)

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA GLOBAL MULTIDIMENSIONAL SOFT-LABEL MODEL RESULTS\n")
    f.write("=" * 80 + "\n\n")

    f.write("1. RUN INFORMATION\n")
    f.write("-" * 80 + "\n")
    f.write(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model name: {model_name}\n")
    f.write(f"Output directory: {output_dir}\n")
    f.write(f"Best validation macro JS: {best_val_js}\n")
    f.write(f"Epochs completed: {len(history_df)}\n\n")

    f.write("2. MODEL OBJECTIVE\n")
    f.write("-" * 80 + "\n")
    f.write("Task: global multidimensional soft-label prediction\n")
    f.write("Architecture: shared RoBERTa encoder with one distribution head per MHS dimension\n")
    f.write("Loss: mean KL divergence across task heads\n\n")

    f.write("Tasks:\n")
    for task, label_values in task_specs.items():
        f.write(f"- {task}: labels {label_values}\n")
    f.write("\n")

    f.write("3. DATASET SIZES\n")
    f.write("-" * 80 + "\n")
    f.write(f"Train rows: {len(train_df)}\n")
    f.write(f"Validation rows: {len(val_df)}\n")
    f.write(f"Test rows: {len(test_df)}\n\n")

    f.write("4. TRAINING SETUP\n")
    f.write("-" * 80 + "\n")
    f.write(f"Max sequence length: {MAX_LENGTH}\n")
    f.write("Train batch size: 8\n")
    f.write("Eval batch size: 16\n")
    f.write(f"Max epochs: {NUM_EPOCHS}\n")
    f.write(f"Learning rate: {LEARNING_RATE}\n")
    f.write(f"Weight decay: {WEIGHT_DECAY}\n")
    f.write(f"Early stopping patience: {PATIENCE}\n")
    f.write(f"Device: {device}\n\n")

    f.write("5. TRAINING HISTORY\n")
    f.write("-" * 80 + "\n")
    f.write(history_df.to_string(index=False))
    f.write("\n\n")

    f.write("6. FINAL VALIDATION METRICS\n")
    f.write("-" * 80 + "\n")
    f.write(val_results_df.to_string(index=False))
    f.write("\n\n")

    f.write("7. FINAL TEST METRICS\n")
    f.write("-" * 80 + "\n")
    f.write(test_results_df.to_string(index=False))
    f.write("\n\n")

    f.write("8. NOTES\n")
    f.write("-" * 80 + "\n")
    f.write("This model introduces multidimensionality without relying on sparse subgroup-specific distributions.\n")
    f.write("All task targets are global annotator distributions over selected MHS survey dimensions.\n")
    f.write("Validation metrics are printed after every epoch and the best model is selected by validation macro JS divergence.\n")

print("Saved report:", results_path)
print("Saved history:", history_path)
print("Saved validation metrics:", val_metrics_path)
print("Saved test metrics:", test_metrics_path)

Saved report: ../results/tables/roberta_global_multidimensional_soft_labels_1_results.txt
Saved history: ../results/tables/roberta_global_multidimensional_soft_labels_1_training_history.csv
Saved validation metrics: ../results/tables/roberta_global_multidimensional_soft_labels_1_validation_metrics.csv
Saved test metrics: ../results/tables/roberta_global_multidimensional_soft_labels_1_test_metrics.csv
